<a href="https://colab.research.google.com/github/veronikaleleka/expense-tracker-ai/blob/main/expense_tracker_ai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install openai

In [ ]:
import io
from PIL import Image
from google.colab import userdata
from google.colab import files
from openai import OpenAI
import json
import pandas as pd
import base64
import mimetypes

In [ ]:
client = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))

In [ ]:
uploaded = files.upload()

In [ ]:
receipts = []

In [ ]:
for filename, content in uploaded.items():
  try:

    mime_type = mimetypes.guess_type(filename)[0] or "image/jpeg"

    img = Image.open(io.BytesIO(content))
    img.thumbnail((1024, 1024))

    buffer = io.BytesIO()

    if img.mode in ("RGBA", "P"):
      img = img.convert("RGB")
    img.save(buffer, format="JPEG", quality=85)

    encoded_image = base64.b64encode(buffer.getvalue()).decode('utf-8')

    response = client.chat.completions.create(
        model='gpt-4o-mini',
        response_format={"type": "json_object"},
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a smart financial assistant. Extract the store name, "
                    "date, total amount, and category (Groceries, Transport, "
                    "Utilities, Dining, Shopping, Other) from the receipt. "
                    "You MUST output the result as a valid JSON object with the "
                    "following keys: store_name, date, total_amount, category."
                )
            },
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": "Extract receipt data."},
                    {
                        "type": "image_url",
                        "image_url": {"url": f"data:{mime_type};base64,{encoded_image}"}
                    }
                ]
            }
        ]
    )

    response_text = response.choices[0].message.content
    json_data = json.loads(response_text)
    json_data['source_file'] = filename
    receipts.append(json_data)
  except Exception as e:
    print(f"Failed to process {filename}: {e}")

In [ ]:
table_data = pd.DataFrame(receipts)
display(table_data)
print(f"Successfully processed: {len(receipts)} from {len(uploaded)}")